In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Load data
df = pd.read_csv('./data/MachineLearningRating_v3.txt', sep='|', engine='python', encoding='utf-8')

print(f"✅ Loaded {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 1036, saw 4


In [ ]:
# Data Quality Report
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Total missing cells: {missing.sum():,}")
print(f"Missing percentage: {(missing.sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"\nColumns with missing values:\n{missing_df}")

In [ ]:
# Calculate Risk Metrics
df['LossRatio'] = np.where(df['TotalPremium'] > 0, df['TotalClaims'] / df['TotalPremium'], 0)
df['ProfitMargin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

print("✅ Calculated: LossRatio, ProfitMargin, HasClaim")

In [ ]:
# Portfolio KPIs
total_premium = df['TotalPremium'].sum()
total_claims = df['TotalClaims'].sum()
loss_ratio = total_claims / total_premium
claim_count = df['HasClaim'].sum()
claim_frequency = (claim_count / len(df)) * 100
claim_severity = df[df['HasClaim'] == 1]['TotalClaims'].mean()
max_claim = df['TotalClaims'].max()

print("=" * 60)
print("PORTFOLIO METRICS")
print("=" * 60)
print(f"Total Premium:     R{total_premium:,.2f}")
print(f"Total Claims:      R{total_claims:,.2f}")
print(f"Profit Margin:     R{total_premium - total_claims:,.2f}")
print(f"Loss Ratio:        {loss_ratio:.4f} ({loss_ratio*100:.2f}%)")
print(f"Claim Count:       {claim_count:,}")
print(f"Claim Frequency:   {claim_frequency:.4f}%")
print(f"Claim Severity:    R{claim_severity:,.2f}")
print(f"Max Claim:         R{max_claim:,.2f}")

In [ ]:
# Descriptive Statistics
numeric_cols = ['TotalPremium', 'TotalClaims', 'SumInsured', 'LossRatio']
available_cols = [col for col in numeric_cols if col in df.columns]

stats = df[available_cols].describe(percentiles=[.25, .5, .75, .95, .99])
stats.loc['skewness'] = df[available_cols].skew()

print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
print(stats)

In [ ]:
# Missing Values Visualization
fig, ax = plt.subplots(figsize=(12, 6))
missing_df['Missing %'].plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_title('Missing Values by Column', fontsize=14)
ax.set_xlabel('Column Name')
ax.set_ylabel('Missing Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: reports/missing_values.png")